# Bloque 1 — Estructuras de Datos, Expresiones Regulares y Funciones Lambda
---
## Proyecto: Red Social Ficticia

Trabajamos con los datos de una **red social ficticia** generada para practicar análisis de datos con Python. Los datos están repartidos en tres ficheros dentro de la carpeta `data/`:

- **`usuarios.csv`** — un registro por usuario, con campos como 
    - `id_usuario` — identificador único del usuario (numérico como string, p. ej. '1'). Se usa para relacionar con publicaciones.csv y conexiones.json.
    - `nombre` — nombre de pila (p. ej. 'Ana').
    - `apellidos` — uno o dos apellidos (p. ej. 'Fernandez Gonzalez').
    - `edad` — edad del usuario en años, como string (p. ej. '24'') → son str(), conviértela con int()
    - `ciudad` — ciudad de residencia; puede venir vacía ('') si el usuario no la indicó.
    - `pais` — país de residencia (en este dataset, siempre 'Espana').
    - `genero` — código de género, 'F' o 'M'.
    - `fecha_registro` — fecha en que el usuario se dio de alta, formato 'YYYY-MM-DD'.
    - `tipo_cuenta` — tipo de cuenta: 'free', 'premium', 'creator', etc.
    - `seguidores` — número de seguidores, como string (p. ej. '2016') → son str(),conviértelo con int()
    - `seguidos` — número de cuentas a las que sigue, también como string.
    - `publicaciones_totales` — número total de publicaciones hechas por el usuario, como string.
    - `activo` — indica si la cuenta está activa, como string 'True'/'False' (compáralo con == 'True').
    - `bio` — texto libre de biografía/descripción del perfil.


- **`publicaciones.csv`** — un registro por publicación, con 

    - `id_publicacion` — identificador único de la publicación (numérico como string).
    - `id_usuario` — id del usuario que hizo la publicación; es la clave para relacionar cada publicación con su autor en usuarios.csv.
    - `fecha` — fecha de publicación, formato 'YYYY-MM-DD' (se puede ordenar directamente como string).
    - `tipo_contenido` — tipo de contenido publicado: 'foto', 'video', 'texto', etc.
    - `likes` — número de "me gusta", como string con formato decimal (p. ej. '1329.0') → conviértelo con float() (no int() directamente, porque tiene el .0).
    - `comentarios` — número de comentarios, también como string decimal (p. ej. '354.0') → igual, usa float().
    - `compartidos` — número de veces compartida, como string entero (p. ej. '105') → puedes usar int().
    - `visualizaciones` — número de visualizaciones, como string entero.
    - `hashtags` — cadena con los hashtags de la publicación separados por comas (p. ej. '#datascience,#cloud'); para trabajar con ellos individualmente hay que hacer .split(','). Puede venir vacía si la publicación no tiene hashtags.

- **`conexiones.json`** — relaciones de seguimiento entre usuarios: una lista de objetos       
    - `id_usuario` — id del usuario (aquí sí es un número entero real, no un string, porque viene de JSON, no de CSV).
    - `siguiendo` — lista de ids (enteros) de los usuarios a los que este usuario sigue. Es decir, representa las relaciones de "seguir" en la red social, una por cada usuario. `{"id_usuario": ..., "siguiendo": [ids de usuarios a los que sigue]}`.

- **Punto clave:**
    
    - Todos los campos numéricos y booleanos llegan como **texto** al leerlos con el módulo `csv` (p. ej. `'2016'` o `'True'`), así que a menudo habrá que convertirlos (`int(...)`) o compararlos como string antes de operar con ellos.
    - Fíjate que `likes` y `comentarios` vienen con decimales `(.0)` mientras que `compartidos` y `visualizaciones` no — así que al convertir usa `float()` para los dos primeros e `int()` para los otros dos (o `int(float(valor)`) si quieres unificar).
    - En el caso de fichero conexiones:
        - Al venir de JSON (no de CSV), los números ya son int, no strings — no hace falta convertirlos con int().
        - No es un diccionario único, sino una lista de diccionarios (uno por usuario), así que para buscar las conexiones de un usuario concreto hay que recorrer la lista y comparar c['id_usuario'] con el id buscado (como se hace en la función siguiendo_de del ejercicio 3.3).
        - Es la estructura que permite calcular cosas como seguidores en común, recomendaciones, o cuántos usuarios siguen a alguien en concreto.

---
### Objetivos
En este bloque trabajamos con los datos de la red social usando **solo la librería estándar de Python** (sin Pandas).

Contenido:
- **Listas**: operaciones, comprensiones, slicing
- **Diccionarios**: índices, agrupaciones, transformaciones
- **Conjuntos**: operaciones de conjunto sobre datos reales
- **Tuplas y namedtuples**: registros inmutables
- **Funciones lambda** con `filter`, `map`, `sorted` y `reduce`
- **Expresiones regulares**: extracción y búsqueda de hashtags con `re`

---
### Bloque de código inicial
Esta celda carga en memoria los tres ficheros de datos de la red social ficticia (usuarios, publicaciones y conexiones) usando solo la librería estándar de Python, sin necesidad de Pandas. 
Busca automáticamente la carpeta `data/` sea cual sea el directorio desde el que se ejecute el notebook, y avisa con un error claro si no encuentra los ficheros. Al final deja listas las variables `usuarios`, `publicaciones` y `conexiones` que se usarán en todos los ejercicios del bloque

In [1]:
from pathlib import Path
import csv
import json
import re
from functools import reduce
from collections import namedtuple

# Carga de datos con el modulo csv estandar (SIN pandas)
posibles_carpetas_datos = [
    Path("../data"),
    Path("data"),
    Path("Proy_analisis_Red_Social_Ficticia/data"),
]

DATA_DIR = next(
    (carpeta for carpeta in posibles_carpetas_datos if (carpeta / "usuarios.csv").exists()),
    None,
)

if DATA_DIR is None:
    rutas_probadas = "\n".join(str(carpeta / "usuarios.csv") for carpeta in posibles_carpetas_datos)
    raise FileNotFoundError(
        "No se encontro usuarios.csv. Rutas probadas:\n"
        f"{rutas_probadas}\n\n"
        f"Directorio actual del kernel: {Path.cwd()}"
    )

def cargar_csv(ruta):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

usuarios = cargar_csv(DATA_DIR / "usuarios.csv")
publicaciones = cargar_csv(DATA_DIR / "publicaciones.csv")

with open(DATA_DIR / "conexiones.json", encoding="utf-8") as f:
    conexiones = json.load(f)

print(f"Carpeta de datos:       {DATA_DIR}")
print(f"Usuarios cargados:      {len(usuarios)}")
print(f"Publicaciones cargadas: {len(publicaciones)}")
print(f"Conexiones cargadas:    {len(conexiones)}")
print(f"\nEjemplo de usuario:")
for k, v in list(usuarios[0].items())[:6]:
    print(f"  {k}: {v}")


Carpeta de datos:       ../data
Usuarios cargados:      100
Publicaciones cargadas: 300
Conexiones cargadas:    100

Ejemplo de usuario:
  id_usuario: 1
  nombre: Ana
  apellidos: Fernandez Gonzalez
  edad: 24
  ciudad: Granada
  pais: Espana


---
## Ejercicios básicos
### Sección 1 — Listas

Las listas son la estructura secuencial mutable más usada en Python.

In [2]:
# ─── Ejercicio 1.1 ─────────────────────────────────────────────────────────
# Practica el bucle 'while' con un índice manual (patrón habitual cuando se
# leen datos con input() hasta que el usuario decide terminar).
# Recorre 'usuarios' con un bucle while y un índice 'i' que empiece en 0.
# Añade el nombre de cada usuario a la lista 'nombres_while' con .append(),
# incrementando 'i' en cada vuelta, hasta procesar los 10 primeros usuarios.

nombres_while = []
i = 0

while i < 10:
    nombres_while.append(usuarios[i]["nombre"])
    i += 1

print(f"Nombres: {nombres_while}")


Nombres: ['Ana', 'Pablo', 'Miguel', 'Pablo', 'Alejandro', 'Elena', 'Raul', 'Daniel', 'Patricia', 'Ivan']


In [3]:
# ─── Ejercicio 1.2 ─────────────────────────────────────────────────────────
# Practica el uso de índices explícitos con range(len(...)).
# Crea una lista de tuplas (posicion, nombre) para los 5 primeros usuarios,
# donde 'posicion' empieza en 0.
# Una vez terminado haz el mismo ejerciio utilizando enumerate() en vez de range(len(...)).

# Solución usando range(len(...))
posiciones_nombres = []
for i in range(5):
    posiciones_nombres.append((i, usuarios[i]['nombre']))

#enumerate() 
posiciones_nombres_enumerate = [(i, u['nombre']) for i, u in enumerate(usuarios) if i < 5]

print(posiciones_nombres)
print(posiciones_nombres_enumerate)


[(0, 'Ana'), (1, 'Pablo'), (2, 'Miguel'), (3, 'Pablo'), (4, 'Alejandro')]
[(0, 'Ana'), (1, 'Pablo'), (2, 'Miguel'), (3, 'Pablo'), (4, 'Alejandro')]


In [ ]:
# ─── Ejercicio 1.3 ─────────────────────────────────────────────────────────
# Practica el patrón de acumulador manual con variable "flag": calcula el
# usuario con MENOS seguidores SIN usar la función min().
# Recorre 'usuarios' con un for y usa una variable booleana 'primero' para
# inicializar el acumulador 'usuario_menos_seguidores' la primera vuelta.


for usuario in usuarios:
    if primero:
        usuario_menos_seguidores = usuario
        primero = False
    elif int(usuario['seguidores']) < int(usuario_menos_seguidores['seguidores']):
        usuario_menos_seguidores = usuario 




In [3]:
# ─── Ejercicio 1.4 ─────────────────────────────────────────────────────────
# Usa SLICING para obtener:
# a) los 5 primeros usuarios
# b) los 10 últimos usuarios
# c) uno de cada tres (cada tercer usuario)

primeros_5  = usuarios[:5]  
ultimos_10  = usuarios[-10:]  
cada_tres   = usuarios[::3] 

print(primeros_5)
print(ultimos_10)
print(cada_tres)

[{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '3', 'nombre': 'Miguel', 'apellidos': 'Ruiz Ruiz', 'edad': '34', 'ciudad': 'Alicante', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-14', 'tipo_cuenta': 'premium', 'seguidores': '802', 'seguidos': '100', 'publicaciones_totales': '282', 'activo': 'True', 'bio': 'Estudiante de Big Data'}, {'id_usuario': '4', 'nombre': 'Pabl

In [4]:
# ─── Ejercicio 1.5 ─────────────────────────────────────────────────────────
# El campo 'activo' viene como string 'True' / 'False' desde el CSV.
# Crea una lista 'activos' con solo los usuarios cuyo campo 'activo' == 'True'.

activos = []

for usuario in usuarios:
    if usuario['activo'] == 'True':
        activos.append(usuario)

print(activos)


[{'id_usuario': '1', 'nombre': 'Ana', 'apellidos': 'Fernandez Gonzalez', 'edad': '24', 'ciudad': 'Granada', 'pais': 'Espana', 'genero': 'F', 'fecha_registro': '2022-05-14', 'tipo_cuenta': 'free', 'seguidores': '2016', 'seguidos': '52', 'publicaciones_totales': '15', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '2', 'nombre': 'Pablo', 'apellidos': 'Fernandez Alvarez', 'edad': '55', 'ciudad': '', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2023-11-30', 'tipo_cuenta': 'premium', 'seguidores': '3446', 'seguidos': '452', 'publicaciones_totales': '174', 'activo': 'True', 'bio': 'Explorando el mundo de la IA'}, {'id_usuario': '3', 'nombre': 'Miguel', 'apellidos': 'Ruiz Ruiz', 'edad': '34', 'ciudad': 'Alicante', 'pais': 'Espana', 'genero': 'M', 'fecha_registro': '2022-02-14', 'tipo_cuenta': 'premium', 'seguidores': '802', 'seguidos': '100', 'publicaciones_totales': '282', 'activo': 'True', 'bio': 'Estudiante de Big Data'}, {'id_usuario': '4', 'nombre': 'Pabl

In [ ]:
# ─── Ejercicio 1.6 ─────────────────────────────────────────────────────────
# Crea una lista 'populares' con los usuarios que tienen MÁS de 1000 seguidores.
# Nota: el campo 'seguidores' viene como string desde el CSV → convierte a int.

populares = []

for usuario in usuarios:
    if int(usuario['seguidores']) > 1000:
        populares.append(usuario)

print(populares)


In [5]:
# ─── Ejercicio 1.7 ─────────────────────────────────────────────────────────
# Crea una lista de tuplas (nombre, seguidores) para todos los usuarios.
# Usa comprensión de lista.

pares_nombre_seg = []
for user in usuarios:
    pares_nombre_seg.append((user['nombre'], int(user['seguidores'])))

print(pares_nombre_seg)


[('Ana', 2016), ('Pablo', 3446), ('Miguel', 802), ('Pablo', 385), ('Alejandro', 5000), ('Elena', 2594), ('Raul', 2162), ('Daniel', 5163), ('Patricia', 2414), ('Ivan', 14035), ('Cristina', 484), ('Ivan', 28730), ('Carlos', 37301), ('Adrian', 4427), ('Maria', 506), ('Laura', 4518), ('Carlos', 2182), ('Raul', 2579), ('Elena', 992), ('Elena', 3252), ('Alejandro', 2869), ('Carmen', 2176), ('Luis', 3479), ('Antonio', 305), ('Raul', 1276), ('Carmen', 327), ('Alejandro', 4183), ('Fernando', 4976), ('Marta', 3543), ('Antonio', 3454), ('Marcos', 1760), ('Pablo', 210), ('Carlos', 3492), ('Beatriz', 48138), ('Elena', 3613), ('Daniel', 4455), ('Sergio', 3204), ('Lucia', 1055), ('Fernando', 3734), ('Luis', 2035), ('Carmen', 2558), ('Carmen', 4928), ('Sergio', 1263), ('Laura', 3702), ('Alejandro', 46952), ('Ana', 2420), ('Isabel', 2069), ('Claudia', 19546), ('Sara', 850), ('Luis', 1232), ('Cristina', 29103), ('Miguel', 31), ('Andrea', 3492), ('Cristina', 1028), ('Ana', 1885), ('Alejandro', 4805), ('N

In [3]:
# ─── Ejercicio 1.8 ─────────────────────────────────────────────────────────
# El campo 'hashtags' de cada publicación es una cadena '#ia,#python,...'
# Crea una lista PLANA con todos los hashtags de todas las publicaciones.
# Incluye los duplicados (se eliminarán en la sección de conjuntos).

todos_hashtags = []
for publicacion in publicaciones:
    hashtags = publicacion['hashtags'].split(',')
    todos_hashtags.extend(hashtags)

print(todos_hashtags)



['#datascience', '#cloud', '#innovacion', '#cloud', '#datascience', '#innovacion', '#analisis', '#bigdata', '#python', '#deeplearning', '#datos', '#deeplearning', '#ia', '#startup', '#developer', '#python', '#analisis', '#bigdata', '#deeplearning', '#ia', '#programacion', '#datos', '#tecnologia', '#innovacion', '#bigdata', '#cloud', '#datos', '#ia', '#bigdata', '#developer', '#startup', '#tecnologia', '#machinelearning', '#developer', '#innovacion', '#python', '#datos', '#tecnologia', '#coding', '#programacion', '#machinelearning', '#python', '#datascience', '#startup', '#machinelearning', '#ia', '#developer', '#tecnologia', '#startup', '#machinelearning', '#cloud', '#developer', '#cloud', '#innovacion', '#datascience', '#tecnologia', '#programacion', '#startup', '#datascience', '#datos', '#ia', '#startup', '#ia', '#bigdata', '#coding', '#python', '#ia', '#innovacion', '#ia', '#datascience', '#bigdata', '#datos', '#innovacion', '#bigdata', '#startup', '#cloud', '#machinelearning', '#de

---
## Sección 2 — Diccionarios

Los diccionarios permiten acceso O(1) por clave y son perfectos para crear índices.

In [9]:
# ─── Ejercicio 2.1 ─────────────────────────────────────────────────────────
# Crea un diccionario 'indice_usuarios' que mapee id_usuario → nombre.
# Así podemos buscar el nombre de un usuario por su id en O(1).

indice_usuarios = {
    usuario['id_usuario']: usuario['nombre']
    for usuario in usuarios
}

print(indice_usuarios)


{'1': 'Ana', '2': 'Pablo', '3': 'Miguel', '4': 'Pablo', '5': 'Alejandro', '6': 'Elena', '7': 'Raul', '8': 'Daniel', '9': 'Patricia', '10': 'Ivan', '11': 'Cristina', '12': 'Ivan', '13': 'Carlos', '14': 'Adrian', '15': 'Maria', '16': 'Laura', '17': 'Carlos', '18': 'Raul', '19': 'Elena', '20': 'Elena', '21': 'Alejandro', '22': 'Carmen', '23': 'Luis', '24': 'Antonio', '25': 'Raul', '26': 'Carmen', '27': 'Alejandro', '28': 'Fernando', '29': 'Marta', '30': 'Antonio', '31': 'Marcos', '32': 'Pablo', '33': 'Carlos', '34': 'Beatriz', '35': 'Elena', '36': 'Daniel', '37': 'Sergio', '38': 'Lucia', '39': 'Fernando', '40': 'Luis', '41': 'Carmen', '42': 'Carmen', '43': 'Sergio', '44': 'Laura', '45': 'Alejandro', '46': 'Ana', '47': 'Isabel', '48': 'Claudia', '49': 'Sara', '50': 'Luis', '51': 'Cristina', '52': 'Miguel', '53': 'Andrea', '54': 'Cristina', '55': 'Ana', '56': 'Alejandro', '57': 'Nuria', '58': 'Ivan', '59': 'Elena', '60': 'Claudia', '61': 'Alicia', '62': 'Alba', '63': 'Sara', '64': 'Sara', '

In [ ]:
# ─── Ejercicio 2.2 ─────────────────────────────────────────────────────────
# Crea un diccionario 'por_ciudad' que mapee ciudad → [lista de nombres].
# Ignora los usuarios sin ciudad (campo vacío o None).

por_ciudad = {}  

for usuario in usuarios:
    ciudad = usuario['ciudad']
    if ciudad:  # Ignorar usuarios sin ciudad
        if ciudad not in por_ciudad:
            por_ciudad[ciudad] = []
        por_ciudad[ciudad].append(usuario['nombre'])

        




In [10]:
# ─── Ejercicio 2.3 ─────────────────────────────────────────────────────────
# Crea una función que reciba un id_usuario (como string) y devuelva
# el diccionario completo del usuario usando el índice creado en 2.1.

def buscar_usuario(id_str):
    """Devuelve el dict del usuario con ese id, o None si no existe."""
    if indice_usuarios.get(id_str) is not None:
        return indice_usuarios.get(id_str)
    else:
        return None
    
buscar_usuario("1")  # Devuelve el dict del usuario con id "1"


'Ana'

In [11]:
# ─── Ejercicio 2.4 ─────────────────────────────────────────────────────────
# Añade un campo calculado 'ratio_engagement' a cada usuario.
# ratio_engagement = seguidores / (seguidos + 1)  (el +1 evita división por cero)
# Trabaja sobre una COPIA de la lista para no modificar el original.

usuarios_con_ratio = []
for u in usuarios:
    u_copia = dict(u)  # copia del diccionario
    u_copia['ratio_engagement'] = int(u['seguidores']) / (int(u['seguidos']) + 1)
    usuarios_con_ratio.append(u_copia)
    usuarios_con_ratio[0]



---
## Sección 3 — Conjuntos

Los conjuntos (`set`) garantizan unicidad y soportan operaciones de teoría de conjuntos.

In [ ]:
# ─── Ejercicio 3.1 ─────────────────────────────────────────────────────────
# Crea un set 'ciudades_unicas' con todas las ciudades distintas de los usuarios.
# Ignora los valores vacíos.

ciudades_unicas = set()
for usuario in usuarios:
    ciudad = usuario['ciudad']
    if ciudad:  # Ignorar valores vacíos
        ciudades_unicas.add(ciudad)
          



In [ ]:
# ─── Ejercicio 3.2 ─────────────────────────────────────────────────────────
# Crea un set 'hashtags_unicos' a partir de la lista 'todos_hashtags' del ejercicio 1.8.
# Muestra cuántos hashtags únicos hay vs el total con duplicados.

hashtags_unicos_set = set(todos_hashtags)




In [ ]:
# ─── Ejercicio 3.3 ─────────────────────────────────────────────────────────
# Usa el fichero de conexiones para encontrar los usuarios seguidos en COMÚN
# por el usuario con id=1 y el usuario con id=2.
# conexiones es una lista de {id_usuario: int, siguiendo: [int, ...]}

def siguiendo_de(id_usuario):
    """Devuelve el set de ids que sigue el usuario dado."""
    for conexion in conexiones:
        if conexion['id_usuario'] == id_usuario:
            return set(conexion['siguiendo'])
    return set()  # Si no se encuentra el usuario, devuelve un set vacío




In [ ]:
# ─── Ejercicio 3.4 ─────────────────────────────────────────────────────────
# Encuentra los usuarios que son seguidos por AL MENOS 3 usuarios distintos.
# Itera sobre todas las conexiones y cuenta cuántas veces aparece cada id.

todos_seguidos = []

populares_seguidos = []
for conexion in conexiones:
    todos_seguidos.extend(conexion['siguiendo'])
    if conexion['id_usuario'] not in populares_seguidos:
        populares_seguidos.append(conexion['id_usuario'])
        



---
## Sección 4 — Tuplas y namedtuples

Las tuplas son inmutables. Las `namedtuple` añaden nombres a los campos, haciendo el código más legible.

In [ ]:
# ─── Ejercicio 4.1 ─────────────────────────────────────────────────────────
# Define un namedtuple 'RegistroUsuario' con los campos:
#   id_usuario, nombre, ciudad, seguidores, tipo_cuenta
# Crea una lista de RegistroUsuario a partir de 'usuarios'.

RegistroUsuario = namedtuple('RegistroUsuario', ['id_usuario', 'nombre', 'ciudad', 'seguidores', 'tipo_cuenta'])

registros = [RegistroUsuario(
    id_usuario=u['id_usuario'],
    nombre=u['nombre'],
    ciudad=u['ciudad'],
    seguidores=int(u['seguidores']),
    tipo_cuenta=u['tipo_cuenta']
) for u in usuarios]



In [ ]:
# ─── Ejercicio 4.2 ─────────────────────────────────────────────────────────
# Demuestra la INMUTABILIDAD de las tuplas:
# Intenta modificar el campo 'seguidores' del primer registro.
# Captura la excepción con try/except y muestra el mensaje de error.

try:
    registros[0].seguidores = 9999  # Intento de modificar un campo de la tupla
except AttributeError as e:
    print(f"Error al intentar modificar un campo de la tupla: {e}")
    

In [ ]:
# ─── Ejercicio 4.3 ─────────────────────────────────────────────────────────
# Usa DESEMPAQUETADO de tupla para extraer los campos del primer registro.
# Asigna cada campo a una variable distinta.

id_u, nombre_u, ciudad_u, seg_u, tipo_u = registros[0]



---
## Sección 5 — Funciones Lambda

`lambda`, `filter`, `map`, `sorted` y `reduce` permiten procesar listas de forma concisa y funcional.

In [ ]:
# ─── Ejercicio 5.1 ─────────────────────────────────────────────────────────
# Ordena 'usuarios' por número de seguidores de MAYOR a MENOR usando sorted() + lambda.

usuarios_por_seg = sorted(usuarios, key=lambda u: int(u['seguidores']), reverse=True)





In [ ]:
# ─── Ejercicio 5.2 ─────────────────────────────────────────────────────────
# Usa filter() + lambda para obtener solo los usuarios de tipo 'creator'.

creators = list(filter(lambda u: u['tipo_cuenta'] == 'creator', usuarios))




In [ ]:
# ─── Ejercicio 5.3 ─────────────────────────────────────────────────────────
# Usa map() + lambda para crear una lista de etiquetas:
#   Si seguidores >= 5000 → 'influencer', si >= 1000 → 'popular', si no → 'normal'

def etiqueta(u):


    seguidores = int(u['seguidores'])
    if seguidores >= 5000:
        return 'influencer'
    elif seguidores >= 1000:
        return 'popular'
    else:
        return 'normal'
     



In [ ]:
# ─── Ejercicio 5.4 ─────────────────────────────────────────────────────────
# Usa reduce() para calcular el TOTAL de seguidores de todos los usuarios.

total_seguidores = reduce(lambda acc, u: acc + int(u['seguidores']), usuarios, 0)




In [ ]:
# ─── Ejercicio 5.5 ─────────────────────────────────────────────────────────
# Ordena las publicaciones por fecha de publicación (de más antigua a más reciente).
# El campo 'fecha' es una cadena 'YYYY-MM-DD' → se puede ordenar como string.

pubs_ordenadas = sorted(publicaciones, key=lambda p: p['fecha'])




---
## Sección 6 — Expresiones regulares

El módulo estándar `re` permite localizar patrones de texto. En una red social resulta útil para extraer etiquetas y buscar publicaciones concretas.

Patrones que usaremos:

```python
r'#[\w]+'             # hashtag: # seguido de letras, números o _
re.findall(patron, texto)
re.search(patron, texto, flags=re.IGNORECASE)
```

Usa cadenas *raw* (`r'...'`) para que las barras invertidas del patrón se interpreten correctamente.

In [ ]:
# ─── Ejercicio R.1 ─────────────────────────────────────────────────────────
# Extrae todos los hashtags de este texto usando re.findall().
# El patrón debe encontrar # seguido de letras, números o _. Normaliza el
# resultado a minúsculas y conserva el orden, incluidos los duplicados.

texto = 'Aprende #Python, #IA y #datos_2026. También #python.'
hashtags_regex = re.findall(r'#\w+', texto)
hashtags_normalizados = [h.lower() for h in hashtags_regex]




In [ ]:
# ─── Ejercicio R.2 ─────────────────────────────────────────────────────────
# Filtra las publicaciones que contienen exactamente el hashtag buscado.
# La búsqueda debe ignorar mayúsculas, pero '#ia' NO debe encontrar '#iabd'.
# Pista: re.escape(hashtag) convierte el hashtag en un patrón seguro y
# (?!\w) evita que después continúen letras, números o _.

hashtag_buscado = '#ia'
patron = re.compile(re.escape(hashtag_buscado) + r'(?!\w)', re.IGNORECASE)
pubs_con_ia = [p for p in publicaciones if patron.search(p['hashtags'])]




: 

---
## Entrega — `../scripts/pipeline.py` (Bloque 1)

Implementa las siguientes funciones en `../scripts/pipeline.py` y ejecuta:
```bash
python -m pytest ../tests/test_bloque1.py -v
```

| Función | Descripción |
|---|---|
| `usuarios_activos(lista)` | Filtra usuarios con `activo == 'True'` |
| `agrupar_por_ciudad(lista)` | Devuelve `{ciudad: [nombres]}` |
| `hashtags_unicos(lista_pub)` | Devuelve `set` de hashtags únicos |
| `top_por_seguidores(lista, n)` | Devuelve los n usuarios con más seguidores |
| `extraer_hashtags_regex(texto)` | Extrae y normaliza hashtags con `re.findall` |
| `publicaciones_con_hashtag(lista_pub, hashtag)` | Filtra por hashtag completo sin distinguir mayúsculas |

## Ejercicios avanzados

Estos ejercicios combinan varias estructuras y funciones. Resuélvelos después de completar los ejercicios básicos.

### A.1 Informe de actividad por ciudad
Crea un diccionario anidado con esta estructura:

```python
{
    'Madrid': {
        'usuarios': 12,
        'activos': 10,
        'seguidores_totales': 25000,
        'media_seguidores': 2083.33
    }
}
```

Agrupa los usuarios por ciudad, ignora ciudades vacías y calcula los cuatro valores usando listas, diccionarios y `lambda`.

### A.2 Recomendaciones por conexiones
Para cada usuario, encuentra hasta cinco recomendaciones: personas que sigue alguno de sus contactos pero que todavía no sigue él. No incluyas al propio usuario y ordena las recomendaciones por número de seguidores.


### A.3 Pipeline funcional de publicaciones
Construye una función que reciba la lista de publicaciones y devuelva las cinco etiquetas más utilizadas. Debe encadenar `map`, `filter`, `sorted` y un diccionario de conteos. Compara el resultado con una solución basada en conjuntos.


### A.4 Limpieza de texto
Crea una función `normalizar_hashtag(texto)` que elimine espacios, convierta a minúsculas y garantice que el resultado comienza por `#`. Aplica la función con `map` y elimina los resultados vacíos o repetidos.

### A.5 Validación
Comprueba que todos los ids de las publicaciones pertenecen a usuarios existentes y devuelve dos listas: publicaciones válidas y publicaciones con errores. No modifiques las listas originales.